In [1]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased', do_lower_case=True)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

In [3]:
#train_set_large = train_set.sample(frac=1).reset_index(drop=True)
large_set = pd.read_csv("995,000_rows.csv").sample(frac=0.01).reset_index(drop=True)
large_set = large_set[["content", "type", "title"]].dropna()

/var/folders/89/bk15wb4x33g0_6ndpf2c8rqh0000gn/T/ipykernel_28370/875026822.py:2: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  large_set = pd.read_csv("995,000_rows.csv").sample(frac=0.01).reset_index(drop=True)


In [4]:
label_map = {
    "fake": "fake",
    "satire": None,
    "bias": None,
    "conspiracy": "fake",
    "state": None,
    "junksci": "fake",
    "hate": None,
    "clickbait": None,
    "unreliable": None,
    "political": "reliable",
    "reliable": "reliable",
    "unknown": None,
}

In [5]:
large_set["new_labels"] = [label_map.get(n, None) for n in large_set["type"]]
#large_set["content_tokens"] = fn.tokenize(large_set["content"])

In [7]:
text_ids = [tokenizer.encode(text, max_length=300, pad_to_max_length=True, truncation=True) for text in large_set["content"]]
att_masks = [[int(id > 0) for id in ids] for ids in text_ids]

/Users/akselrasmussen/uni/ds/fake_news/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [76]:
from sklearn.model_selection import train_test_split

labels = [1.0 if n == "fake" else -1.0 for n in large_set["new_labels"]]

train_x, test_val_x, train_y, test_val_y = train_test_split(text_ids, labels, test_size=0.2)
train_m, test_val_m = train_test_split(att_masks, test_size=0.2)

test_x, val_x, test_y, val_y = train_test_split(test_val_x, test_val_y, test_size=0.5)
test_m, val_m = train_test_split(test_val_m, test_size=0.5)

In [77]:
train_x = torch.tensor(train_x)
test_x = torch.tensor(test_x)
val_x = torch.tensor(val_x)

train_y = torch.tensor(train_y)
test_y = torch.tensor(test_y)
val_y = torch.tensor(val_y)

train_m = torch.tensor(train_m)
test_m = torch.tensor(test_m)
val_m = torch.tensor(val_m)

In [80]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

batch_size = 32

train_data = TensorDataset(train_x, train_m, train_y)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

val_data = TensorDataset(val_x, val_m, val_y)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler=val_sampler, batch_size=batch_size)

In [ ]:
encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=4)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=3)
src = torch.rand(13, 31, 512)
out = transformer_encoder.forward(src)
print(out.shape)

In [ ]:
max_seq_len = max([len(n) for n in train_set["content"]])

In [ ]:
embeddings = nn.Embedding(num_embeddings=dict_size+special_tokens, embedding_dim=512, padding_idx=0)
proj_out = nn.Linear(512, dict_size+special_tokens)
positional_encoding = nn.Parameter(torch.zeros(max_seq_len, 512))

In [ ]:
def forward(seq: int):
    seq = seq[:max_seq_len]
    x = embeddings(seq)
    x += positional_encoding[:len(seq), :]
    return proj_out(transformer_encoder(x))

In [ ]:
# Pre training ...

opt = torch.optim.Adam([*transformer_encoder.parameters(), *embeddings.parameters(), *proj_out.parameters(), positional_encoding], lr=0.001)
costs = []

for epoch in range(3):
    n = 0

    for seq, label in zip(train_set["content"], train_set["new_labels"]):
        if type(seq) is not str:
            print(seq)
            continue
        seq = torch.LongTensor(tokenizer.encode(seq).ids)[:max_seq_len] + special_tokens
        y_seq = seq.clone()
        #y_seq = F.one_hot(seq, num_classes=501).to(torch.int64)
        seq[random.randint(0, len(seq)-1)] = 0
        out = forward(seq)
        loss = F.cross_entropy(out, y_seq)
        loss.backward()
        opt.step()
        opt.zero_grad()
        costs.append(loss.item())
        print(f"{epoch} : {n} / {len(train_set)} - {float(np.mean(costs)):.2f}", end='\r')
        n+=1
        #if float(np.mean(costs)) < 1.1:
        #    break

In [ ]:
proj_out = nn.Linear(512, 1)
def forward(seq: int):
    seq = seq[:max_seq_len]
    x = embeddings(seq)
    x += positional_encoding[:len(seq), :]
    x = torch.vstack([x, torch.zeros([512])])
    #return proj_out(transformer_encoder(x).mean(axis=0)).sigmoid()
    return proj_out(transformer_encoder(x)[-1,:]).tanh()

opt = torch.optim.Adam([*transformer_encoder.parameters(), *embeddings.parameters(), positional_encoding], lr=0.0001)
opt2 = torch.optim.Adam(proj_out.parameters(), lr=0.001)
costs = []

for epoch in range(2):
    n = 0

    for seq, label in zip(train_set["content"], train_set["new_labels"]):
        if type(seq) is not str:
            print(seq)
            continue
        seq = torch.LongTensor(tokenizer.encode(seq).ids)[:max_seq_len] + special_tokens
        #seq[random.randint(0, len(seq))] = 0
    
        if label == "fake":
            y = 1
        else:
            y = -1
        
        out = forward(seq)
        loss = F.mse_loss(out, torch.Tensor([y]))
        costs.append(loss.item())
        loss.backward()
        opt2.step()
        opt2.zero_grad()
        opt.step()
        opt.zero_grad()
        print(f"{epoch} : {n} / {len(train_set.index)} - {float(np.mean(costs)):.2f} - {out.item():.2f} - {label}", end='\r')
        n += 1
        #if n > 130:
        #    break

In [ ]:
true_positive = 0
false_negative = 0
correct = 0

for seq, label in zip(train_set["content"], train_set["new_labels"]):
    seq = torch.LongTensor(tokenizer.encode(seq).ids)[:max_seq_len] + special_tokens
    
    y_hat = "fake" if forward(seq).item() > 0.0 else "reliable"
    
    if y_hat == "fake" and label == "fake":
        true_positive += 1
    if y_hat == "fake" and label == "reliable":
        false_negative += 1
    if y_hat == label:
        correct += 1

#print(f"F1: {true_positive / (true_positive + false_negative) * 100}%")
print(f"accuracy: {correct / len(train_set) * 100}%")

In [ ]:
n = 0
for seq, label in zip(train_set["content"], train_set["new_labels"]):
    seq = torch.LongTensor(tokenizer.encode(seq).ids)[:max_seq_len] + special_tokens
    out = forward(seq).item()
    print(f"{out} - {label}")
    if n > 10:
        break
    n+=1